ML

LAB04

Bài tập 1

In [2]:
# -*- coding: utf-8 -*-
"""Bài 1: Tự viết KNN không dùng thư viện."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Dự đoán nhãn cho một mẫu mới x_moi."""
    # Tính khoảng cách Euclid từ x_moi đến toàn bộ X_hoc
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))

    # Lấy chỉ số của k mẫu có khoảng cách nhỏ nhất
    chi_so_gan = np.argsort(khoang_cach)[:k]

    # Lấy nhãn của k mẫu đó
    nhan_lang_gieng = y_hoc[chi_so_gan]

    # Bỏ phiếu (vì nhãn là 0 và 1)
    dem = np.bincount(nhan_lang_gieng)
    return np.argmax(dem)

# Đọc dữ liệu hai vầng trăng
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy()
y = df["nhan"].to_numpy()

# Chia tập dữ liệu (giống code mẫu)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

k = 13

# 1. Dự đoán bằng hàm tự viết
y_du_doan_tu_viet = np.array([knn_du_doan(X_tr, y_tr, x, k) for x in X_te])
acc_tu_viet = accuracy_score(y_te, y_du_doan_tu_viet)

# 2. Dự đoán bằng scikit-learn
knn_sklearn = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, knn_sklearn.predict(X_te))

print(f"Accuracy (tu viet) : {acc_tu_viet:.4f}")
print(f"Accuracy (sklearn) : {acc_sklearn:.4f}")

Accuracy (tu viet) : 0.9200
Accuracy (sklearn) : 0.9200


Bài tập 2:

In [3]:
# -*- coding: utf-8 -*-
"""Bài 2: Khoảng cách Manhattan."""
import pandas as pd
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

# Tìm k tốt nhất bằng kiểm định chéo với khoảng cách Manhattan
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k, metric="manhattan"),
                              X_tr, y_tr, cv=5).mean()

k_tot = max(diem, key=diem.get)
mh = KNeighborsClassifier(n_neighbors=k_tot, metric="manhattan").fit(X_tr, y_tr)
acc_kt = accuracy_score(y_te, mh.predict(X_te))

print(f"Khoang cach Manhattan: k tot nhat = {k_tot}, Accuracy = {acc_kt:.4f}")

Khoang cach Manhattan: k tot nhat = 27, Accuracy = 0.9067


Bài tập 3:

In [4]:
# -*- coding: utf-8 -*-
"""Bài 3: Đổi đơn vị đo cột chi tiêu."""
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_mo_rong.csv")
# Đổi nghìn đồng sang triệu đồng
df["chi_tieu"] = df["chi_tieu"] / 1000.0
y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
X_tr, X_te, y_tr, y_te = train_test_split(X3, y, test_size=0.25, random_state=17, stratify=y)

# 1. KNN không chuẩn hoá
knn = KNeighborsClassifier(n_neighbors=7).fit(X_tr, y_tr)
acc_khong_chuan_hoa = accuracy_score(y_te, knn.predict(X_te))

# 2. KNN có chuẩn hoá (dùng pipeline)
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)).fit(X_tr, y_tr)
acc_co_chuan_hoa = accuracy_score(y_te, pipe.predict(X_te))

print(f"Khong chuan hoa (don vi trieu): {acc_khong_chuan_hoa:.4f}")
print(f"Co chuan hoa (don vi trieu)  : {acc_co_chuan_hoa:.4f}")

Khong chuan hoa (don vi trieu): 0.8667
Co chuan hoa (don vi trieu)  : 0.8667


Bài tập 4:

In [5]:
# -*- coding: utf-8 -*-
"""Bài 4: Láng giềng có trọng số."""
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

for k in [1, 13, 101]:
    mh = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"k = {k:3d}, weights='distance': {acc:.4f}")

k =   1, weights='distance': 0.8000
k =  13, weights='distance': 0.9333
k = 101, weights='distance': 0.9067


Bài tập 5:

In [8]:
# -*- coding: utf-8 -*-
"""Bài 5: Chọn k cho bài toán hồi quy."""
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Tìm k tốt nhất bằng cross-validation
diem = {}
for k in range(1, 21):
    # scoring âm MAE, nên điểm càng lớn (gần 0) càng tốt
    mae_am = cross_val_score(KNeighborsRegressor(n_neighbors=k),
                             X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error")
    diem[k] = -mae_am.mean()  # Đổi dấu để được MAE dương

k_tot = min(diem, key=diem.get)
print(f"k tot nhat = {k_tot}, MAE CV = {diem[k_tot]:.4f}")

# Đánh giá trên tập kiểm tra
knn = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))

duong = LinearRegression().fit(X_tr, y_tr)
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print(f"MAE KNN (k={k_tot}) : {mae_knn:.4f}")
print(f"MAE Duong thang  : {mae_duong:.4f}")

k tot nhat = 2, MAE CV = 0.3498
MAE KNN (k=2) : 0.4867
MAE Duong thang  : 0.2578


Bài tập 6:

In [9]:
# -*- coding: utf-8 -*-
"""Bài 6: Đo thời gian dự đoán."""
import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
sizes = [1000, 10000, 100000]
X_moi = rng.random((1000, 2)) # 1000 mẫu mới cần dự đoán

print(f"{'So mau':>10} | {'Thoi gian fit (s)':>20} | {'Thoi gian predict (s)':>20}")
print("-" * 60)

for n in sizes:
    X = rng.random((n, 2))
    y = rng.integers(0, 2, n) # Tạo nhãn 0, 1 ngẫu nhiên
    knn = KNeighborsClassifier(n_neighbors=5)

    t0 = time.time()
    knn.fit(X, y)
    t_fit = time.time() - t0

    t0 = time.time()
    knn.predict(X_moi)
    t_predict = time.time() - t0

    print(f"{n:>10} | {t_fit:>20.6f} | {t_predict:>20.6f}")

    So mau |    Thoi gian fit (s) | Thoi gian predict (s)
------------------------------------------------------------
      1000 |             0.002746 |             0.006478
     10000 |             0.006397 |             0.004954
    100000 |             0.092913 |             0.007084
